# 03.1 - Most-Asked Interview Problems by Pattern

You've learned the data structures (`01`) and algorithms (`02`). Now put them together: **44 classic interview problems grouped into 12 patterns**, followed by cheat sheets and practice problems.

**Part 1 - Patterns + Problems:** each problem has its LeetCode number, difficulty, statement → **Idea** → code → complexity → tests.
**Part 2 - Cheat Sheets:** how to recognise which pattern a question needs, Big-O reference, interview routine.
**Part 3 - Practice:** unsolved problems with tests, for you to solve.

### How to study each problem
1. Read the statement and **try it yourself first**.
2. Read the **Idea** - remember the idea, not the code.
3. Run, change inputs, add prints.
4. Re-solve on LeetCode from memory 2–3 days later.

> Every problem has `assert` tests. If the cell runs without error, the solution is correct.

## Setup - classes LeetCode provides (+ test helpers)

In [1]:
from typing import List, Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# ---------- helpers for testing (not needed on LeetCode) ----------
def build_list(values):
    dummy = ListNode()
    cur = dummy
    for v in values:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def list_to_py(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

def build_tree(values):
    # level-order list like LeetCode: [3,9,20,None,None,15,7]
    if not values or values[0] is None:
        return None
    from collections import deque
    root = TreeNode(values[0])
    queue, i = deque([root]), 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); queue.append(node.right)
        i += 1
    return root

def tree_to_list(root):
    from collections import deque
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            out.append(node.val); q.append(node.left); q.append(node.right)
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

print(list_to_py(build_list([1, 2, 3])))
print(tree_to_list(build_tree([3, 9, 20, None, None, 15, 7])))

[1, 2, 3]
[3, 9, 20, None, None, 15, 7]


---
# Part 1 - Patterns & Most-Asked Problems

Each problem: **LeetCode #, difficulty, statement → Idea → Code → Complexity → Tests.**
Problems are ordered so each one builds on the previous.

## Pattern 1 - Arrays & Hashing
**Recognise it:** "find pair / duplicate / count / group", "in O(n)". Trade memory for speed with a dict or set.

### 1. Two Sum - LC #1 (Easy) most asked question ever
Given `nums` and `target`, return indices of the two numbers that add up to `target`.

**Idea:** For each number `x`, the partner we need is `target - x`. Store every number we've seen in a dict `value → index`; if the partner is already there, done. One pass.

In [2]:
def two_sum(nums, target):
    seen = {}                      # value -> index
    for i, x in enumerate(nums):
        need = target - x
        if need in seen:
            return [seen[need], i]
        seen[x] = i
    return []

# Time O(n), Space O(n)   (brute force double loop = O(n^2))
assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
assert two_sum([3, 3], 6) == [0, 1]
print("Two Sum ")

Two Sum 


### 2. Contains Duplicate - LC #217 (Easy)
Return `True` if any value appears at least twice.

**Idea:** a set only keeps unique values → compare sizes (or check `in seen` while looping to exit early).

In [3]:
def contains_duplicate(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False
    # one-liner: return len(set(nums)) != len(nums)

# Time O(n), Space O(n)
assert contains_duplicate([1, 2, 3, 1]) is True
assert contains_duplicate([1, 2, 3, 4]) is False
print("Contains Duplicate ")

Contains Duplicate 


### 3. Valid Anagram - LC #242 (Easy)
Are `s` and `t` anagrams (same letters, same counts)?

**Idea:** compare frequency maps. `Counter(s) == Counter(t)`. (Sorting also works: O(n log n).)

In [4]:
from collections import Counter

def is_anagram(s, t):
    return Counter(s) == Counter(t)

def is_anagram_array(s, t):        # what interviewers may ask: no Counter, 26-letter array
    if len(s) != len(t):
        return False
    count = [0] * 26
    for a, b in zip(s, t):
        count[ord(a) - ord('a')] += 1
        count[ord(b) - ord('a')] -= 1
    return all(c == 0 for c in count)

# Time O(n), Space O(1) (max 26 letters)
assert is_anagram("anagram", "nagaram") and is_anagram_array("anagram", "nagaram")
assert not is_anagram("rat", "car") and not is_anagram_array("rat", "car")
print("Valid Anagram ")

Valid Anagram 


### 4. Group Anagrams - LC #49 (Medium)
Group words that are anagrams of each other.

**Idea:** anagrams share the same **sorted** form → use it as a dict key. `"eat" → "aet"`, `"tea" → "aet"`.

In [5]:
from collections import defaultdict

def group_anagrams(strs):
    groups = defaultdict(list)
    for word in strs:
        key = "".join(sorted(word))   # or tuple of 26 counts -> O(k) instead of O(k log k)
        groups[key].append(word)
    return list(groups.values())

# Time O(n * k log k), k = max word length
result = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
assert sorted(sorted(g) for g in result) == [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]
print("Group Anagrams ", result)

Group Anagrams  [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


### 5. Top K Frequent Elements - LC #347 (Medium)
Return the `k` most frequent elements.

**Idea A:** `Counter` + `heapq.nlargest` → O(n log k).
**Idea B (bucket sort, O(n)):** index = frequency; bucket[f] holds numbers appearing f times; walk buckets from high to low.

In [6]:
from collections import Counter
import heapq

def top_k_frequent(nums, k):
    count = Counter(nums)
    return heapq.nlargest(k, count.keys(), key=count.get)

def top_k_frequent_bucket(nums, k):
    count = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for num, freq in count.items():
        buckets[freq].append(num)
    result = []
    for freq in range(len(buckets) - 1, 0, -1):
        for num in buckets[freq]:
            result.append(num)
            if len(result) == k:
                return result
    return result

assert sorted(top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == [1, 2]
assert sorted(top_k_frequent_bucket([1, 1, 1, 2, 2, 3], 2)) == [1, 2]
assert top_k_frequent([1], 1) == [1]
print("Top K Frequent ")

Top K Frequent 


### 6. Product of Array Except Self - LC #238 (Medium)
`answer[i]` = product of all elements except `nums[i]`, **without division**, O(n).

**Idea (prefix/suffix):** answer[i] = (product of everything on the left) × (product of everything on the right). One pass left→right, one pass right→left.

In [7]:
def product_except_self(nums):
    n = len(nums)
    answer = [1] * n
    prefix = 1
    for i in range(n):              # answer[i] = product of left side
        answer[i] = prefix
        prefix *= nums[i]
    suffix = 1
    for i in range(n - 1, -1, -1):  # multiply by product of right side
        answer[i] *= suffix
        suffix *= nums[i]
    return answer

# Time O(n), Space O(1) extra (output not counted)
assert product_except_self([1, 2, 3, 4]) == [24, 12, 8, 6]
assert product_except_self([-1, 1, 0, -3, 3]) == [0, 0, 9, 0, 0]
print("Product Except Self ")

Product Except Self 


### 7. Longest Consecutive Sequence - LC #128 (Medium)
Length of the longest run of consecutive integers (unsorted input), in O(n).

**Idea:** put everything in a set. Only start counting from a number `x` if `x - 1` is **not** in the set (it's the start of a run). Then count up `x+1, x+2…`.

In [8]:
def longest_consecutive(nums):
    num_set = set(nums)
    best = 0
    for x in num_set:
        if x - 1 not in num_set:        # x is the start of a sequence
            length = 1
            while x + length in num_set:
                length += 1
            best = max(best, length)
    return best

# Time O(n): each number is visited by the while-loop at most once
assert longest_consecutive([100, 4, 200, 1, 3, 2]) == 4
assert longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]) == 9
assert longest_consecutive([]) == 0
print("Longest Consecutive ")

Longest Consecutive 


## Pattern 2 - Two Pointers
**Recognise it:** sorted array, palindromes, "pair with sum", reverse in place. One pointer at each end moving inward (or slow/fast pointers). Turns O(n²) into O(n).

### 8. Valid Palindrome - LC #125 (Easy)
Ignoring non-alphanumeric chars and case, is `s` a palindrome?

In [9]:
def is_palindrome(s):
    l, r = 0, len(s) - 1
    while l < r:
        if not s[l].isalnum():
            l += 1
        elif not s[r].isalnum():
            r -= 1
        else:
            if s[l].lower() != s[r].lower():
                return False
            l += 1
            r -= 1
    return True

# Time O(n), Space O(1)
assert is_palindrome("A man, a plan, a canal: Panama")
assert not is_palindrome("race a car")
assert is_palindrome(" ")
print("Valid Palindrome ")

Valid Palindrome 


### 9. Two Sum II (sorted input) - LC #167 (Medium)
Same as Two Sum but the array is **sorted**; return 1-based indices using O(1) space.

**Idea:** sum too small → move left pointer right; too big → move right pointer left.

In [10]:
def two_sum_sorted(numbers, target):
    l, r = 0, len(numbers) - 1
    while l < r:
        s = numbers[l] + numbers[r]
        if s == target:
            return [l + 1, r + 1]
        if s < target:
            l += 1
        else:
            r -= 1
    return []

assert two_sum_sorted([2, 7, 11, 15], 9) == [1, 2]
assert two_sum_sorted([2, 3, 4], 6) == [1, 3]
print("Two Sum II ")

Two Sum II 


### 10. 3Sum - LC #15 (Medium) All unique triplets that sum to 0.

**Idea:** sort; fix `nums[i]`, then run Two Sum II (two pointers) on the rest. Skip duplicates to avoid repeated triplets.

In [11]:
def three_sum(nums):
    nums.sort()
    res = []
    for i in range(len(nums) - 2):
        if i > 0 and nums[i] == nums[i - 1]:     # skip duplicate first numbers
            continue
        l, r = i + 1, len(nums) - 1
        while l < r:
            s = nums[i] + nums[l] + nums[r]
            if s < 0:
                l += 1
            elif s > 0:
                r -= 1
            else:
                res.append([nums[i], nums[l], nums[r]])
                l += 1
                while l < r and nums[l] == nums[l - 1]:   # skip duplicates
                    l += 1
    return res

# Time O(n^2), Space O(1) extra (ignoring sort)
assert three_sum([-1, 0, 1, 2, -1, -4]) == [[-1, -1, 2], [-1, 0, 1]]
assert three_sum([0, 0, 0, 0]) == [[0, 0, 0]]
assert three_sum([0, 1, 1]) == []
print("3Sum ")

3Sum 


### 11. Container With Most Water - LC #11 (Medium)
Pick two lines that hold the most water: area = `min(h[l], h[r]) * (r - l)`.

**Idea:** start widest; always move the **shorter** line inward (moving the taller can never help).

In [12]:
def max_area(height):
    l, r = 0, len(height) - 1
    best = 0
    while l < r:
        best = max(best, min(height[l], height[r]) * (r - l))
        if height[l] < height[r]:
            l += 1
        else:
            r -= 1
    return best

assert max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49
assert max_area([1, 1]) == 1
print("Container With Most Water ")

Container With Most Water 


## Pattern 3 - Sliding Window
**Recognise it:** "longest / shortest **substring / subarray** such that…", "max sum of k consecutive". Keep a window `[l, r]`; expand `r`, shrink `l` when the window breaks the rule.

### 12. Best Time to Buy and Sell Stock - LC #121 (Easy) Buy one day, sell a later day; max profit.

**Idea:** track the cheapest price so far; profit today = price − cheapest.

In [13]:
def max_profit(prices):
    min_price = float('inf')
    best = 0
    for p in prices:
        min_price = min(min_price, p)
        best = max(best, p - min_price)
    return best

# Time O(n), Space O(1)
assert max_profit([7, 1, 5, 3, 6, 4]) == 5
assert max_profit([7, 6, 4, 3, 1]) == 0
print("Stock ")

Stock 


### 13. Longest Substring Without Repeating Characters - LC #3 (Medium) **Idea:** window holds unique chars. Remember the last index of each char; if the new char is already inside the window, jump `l` past its previous position.

In [14]:
def length_of_longest_substring(s):
    last = {}          # char -> last index
    l = 0
    best = 0
    for r, ch in enumerate(s):
        if ch in last and last[ch] >= l:
            l = last[ch] + 1          # shrink window past the duplicate
        last[ch] = r
        best = max(best, r - l + 1)
    return best

# Time O(n), Space O(min(n, alphabet))
assert length_of_longest_substring("abcabcbb") == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("pwwkew") == 3
assert length_of_longest_substring("") == 0
print("Longest Substring ")

Longest Substring 


### 14. Longest Repeating Character Replacement - LC #424 (Medium)
You may replace at most `k` chars. Longest substring of one repeated letter?

**Idea:** window is valid if `window_length - count_of_most_frequent_char <= k`. Otherwise shrink from the left.

In [15]:
from collections import defaultdict

def character_replacement(s, k):
    count = defaultdict(int)
    l = 0
    max_freq = 0
    best = 0
    for r, ch in enumerate(s):
        count[ch] += 1
        max_freq = max(max_freq, count[ch])
        while (r - l + 1) - max_freq > k:
            count[s[l]] -= 1
            l += 1
        best = max(best, r - l + 1)
    return best

assert character_replacement("ABAB", 2) == 4
assert character_replacement("AABABBA", 1) == 4
print("Character Replacement ")

Character Replacement 


## Pattern 4 - Stack
**Recognise it:** matching brackets, "undo", evaluate expressions, **"next greater / smaller element"** (monotonic stack). In Python a list is a stack: `append`, `pop`, `stack[-1]`.

### 15. Valid Parentheses - LC #20 (Easy) 

In [16]:
def is_valid(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = []
    for ch in s:
        if ch in pairs:                       # closing bracket
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
        else:                                 # opening bracket
            stack.append(ch)
    return not stack                          # nothing left open

assert is_valid("()[]{}") and is_valid("{[()]}")
assert not is_valid("(]") and not is_valid("([)]") and not is_valid("(")
print("Valid Parentheses ")

Valid Parentheses 


### 16. Min Stack - LC #155 (Medium)
Stack supporting `push, pop, top, getMin` all in O(1).

**Idea:** store pairs `(value, min_so_far)` - the minimum is always on top.

In [17]:
class MinStack:
    def __init__(self):
        self.stack = []        # (value, current_min)

    def push(self, val):
        current_min = min(val, self.stack[-1][1]) if self.stack else val
        self.stack.append((val, current_min))

    def pop(self):
        self.stack.pop()

    def top(self):
        return self.stack[-1][0]

    def getMin(self):
        return self.stack[-1][1]

ms = MinStack()
ms.push(-2); ms.push(0); ms.push(-3)
assert ms.getMin() == -3
ms.pop()
assert ms.top() == 0 and ms.getMin() == -2
print("Min Stack ")

Min Stack 


### 17. Daily Temperatures - LC #739 (Medium) - monotonic stack
For each day, how many days until a warmer temperature?

**Idea:** keep a stack of indices with **decreasing** temperatures. When a warmer day arrives, pop all colder days and fill their answers.

In [18]:
def daily_temperatures(temps):
    answer = [0] * len(temps)
    stack = []                       # indices, temps decreasing
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            answer[j] = i - j
        stack.append(i)
    return answer

# Time O(n): each index pushed & popped once
assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
assert daily_temperatures([30, 40, 50, 60]) == [1, 1, 1, 0]
print("Daily Temperatures ")

Daily Temperatures 


## Pattern 5 - Binary Search
**Recognise it:** **sorted** input, "find in O(log n)", or "find the minimum value that works" (binary search on the answer). Halve the search space each step.

### 18. Binary Search - LC #704 (Easy) - memorise this template

In [19]:
def binary_search(nums, target):
    l, r = 0, len(nums) - 1
    while l <= r:
        mid = (l + r) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            l = mid + 1
        else:
            r = mid - 1
    return -1

# Time O(log n)
assert binary_search([-1, 0, 3, 5, 9, 12], 9) == 4
assert binary_search([-1, 0, 3, 5, 9, 12], 2) == -1
print("Binary Search ")

Binary Search 


### 19. Search in Rotated Sorted Array - LC #33 (Medium) e.g. `[4,5,6,7,0,1,2]`. Find `target` in O(log n).

**Idea:** at every `mid`, **one half is sorted**. Check if target lies inside the sorted half; if yes search there, otherwise search the other half.

In [20]:
def search_rotated(nums, target):
    l, r = 0, len(nums) - 1
    while l <= r:
        mid = (l + r) // 2
        if nums[mid] == target:
            return mid
        if nums[l] <= nums[mid]:                    # left half sorted
            if nums[l] <= target < nums[mid]:
                r = mid - 1
            else:
                l = mid + 1
        else:                                       # right half sorted
            if nums[mid] < target <= nums[r]:
                l = mid + 1
            else:
                r = mid - 1
    return -1

assert search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert search_rotated([4, 5, 6, 7, 0, 1, 2], 3) == -1
assert search_rotated([1], 0) == -1
print("Search Rotated ")

Search Rotated 


### 20. Koko Eating Bananas - LC #875 (Medium) - binary search on the answer
Min eating speed `k` to finish all piles within `h` hours.

**Idea:** speed ranges from 1 to `max(piles)`. If speed `k` works, any bigger speed works too → binary search for the smallest speed that works.

In [21]:
def min_eating_speed(piles, h):
    def hours_needed(k):
        return sum((p + k - 1) // k for p in piles)   # ceil(p / k)

    l, r = 1, max(piles)
    while l < r:
        mid = (l + r) // 2
        if hours_needed(mid) <= h:
            r = mid          # mid works -> try smaller
        else:
            l = mid + 1      # too slow
    return l

# Time O(n log max(piles))
assert min_eating_speed([3, 6, 7, 11], 8) == 4
assert min_eating_speed([30, 11, 23, 4, 20], 5) == 30
print("Koko ")

Koko 


## Pattern 6 - Linked List
**Recognise it:** `ListNode`. Tools: **dummy node** (avoids head edge cases), **prev/curr pointer reversal**, **slow/fast pointers** (middle, cycles).

### 21. Reverse Linked List - LC #206 (Easy) 

In [22]:
def reverse_list(head):
    prev, curr = None, head
    while curr:
        nxt = curr.next       # save next
        curr.next = prev      # reverse the pointer
        prev = curr           # move forward
        curr = nxt
    return prev

def reverse_list_recursive(head):
    if not head or not head.next:
        return head
    new_head = reverse_list_recursive(head.next)
    head.next.next = head
    head.next = None
    return new_head

assert list_to_py(reverse_list(build_list([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
assert list_to_py(reverse_list_recursive(build_list([1, 2]))) == [2, 1]
assert reverse_list(None) is None
print("Reverse List ")

Reverse List 


### 22. Merge Two Sorted Lists - LC #21 (Easy)
**Idea:** dummy node + tail pointer; always attach the smaller head.

In [23]:
def merge_two_lists(l1, l2):
    dummy = ListNode()
    tail = dummy
    while l1 and l2:
        if l1.val <= l2.val:
            tail.next, l1 = l1, l1.next
        else:
            tail.next, l2 = l2, l2.next
        tail = tail.next
    tail.next = l1 or l2          # attach the remainder
    return dummy.next

assert list_to_py(merge_two_lists(build_list([1, 2, 4]), build_list([1, 3, 4]))) == [1, 1, 2, 3, 4, 4]
assert list_to_py(merge_two_lists(None, build_list([0]))) == [0]
print("Merge Two Lists ")

Merge Two Lists 


### 23. Linked List Cycle - LC #141 (Easy) - Floyd's slow/fast pointers
**Idea:** slow moves 1 step, fast moves 2. If there's a cycle, fast eventually laps slow.

In [24]:
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

# Time O(n), Space O(1)
head = build_list([3, 2, 0, -4])
head.next.next.next.next = head.next      # create a cycle: -4 -> 2
assert has_cycle(head)
assert not has_cycle(build_list([1, 2]))
print("Linked List Cycle ")

Linked List Cycle 


### 24. Remove Nth Node From End - LC #19 (Medium)
**Idea:** move `fast` n steps ahead, then move both until `fast` reaches the end → `slow` is just before the node to delete. Dummy node handles deleting the head.

In [25]:
def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    slow = fast = dummy
    for _ in range(n):
        fast = fast.next
    while fast.next:
        slow, fast = slow.next, fast.next
    slow.next = slow.next.next
    return dummy.next

assert list_to_py(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
assert list_to_py(remove_nth_from_end(build_list([1]), 1)) == []
print("Remove Nth ")

Remove Nth 


## Pattern 7 - Trees (DFS & BFS)
**Recognise it:** `TreeNode`. Almost every tree problem is:
- **DFS (recursion):** solve for left child, solve for right child, combine. Base case: `if not node`.
- **BFS (deque):** level by level.

### 25. Maximum Depth of Binary Tree - LC #104 (Easy) 

In [26]:
def max_depth(root):
    if not root:
        return 0
    return 1 + max(max_depth(root.left), max_depth(root.right))

# Time O(n), Space O(h) recursion stack (h = tree height)
assert max_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 3
assert max_depth(build_tree([])) == 0
print("Max Depth ")

Max Depth 


### 26. Invert Binary Tree - LC #226 (Easy)

In [27]:
def invert_tree(root):
    if not root:
        return None
    root.left, root.right = invert_tree(root.right), invert_tree(root.left)
    return root

assert tree_to_list(invert_tree(build_tree([4, 2, 7, 1, 3, 6, 9]))) == [4, 7, 2, 9, 6, 3, 1]
print("Invert Tree ")

Invert Tree 


### 27. Binary Tree Level Order Traversal - LC #102 (Medium) - BFS template **Idea:** queue; process exactly `len(queue)` nodes per level.

In [28]:
from collections import deque

def level_order(root):
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):          # nodes of current level only
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.append(level)
    return result

assert level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]
assert level_order(None) == []
print("Level Order ")

Level Order 


### 28. Validate Binary Search Tree - LC #98 (Medium) **Idea:** every node must lie in a range `(low, high)` inherited from its ancestors - checking only the direct children is a classic mistake.

In [29]:
def is_valid_bst(root, low=float('-inf'), high=float('inf')):
    if not root:
        return True
    if not (low < root.val < high):
        return False
    return (is_valid_bst(root.left, low, root.val) and
            is_valid_bst(root.right, root.val, high))

assert is_valid_bst(build_tree([2, 1, 3]))
assert not is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6]))
assert not is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7]))   # 3 < 5 in right subtree
print("Validate BST ")

Validate BST 


### 29. Lowest Common Ancestor of a BST - LC #235 (Medium)
**Idea:** if both values are smaller → go left; both bigger → go right; otherwise the current node splits them → it's the LCA.

In [30]:
def lowest_common_ancestor(root, p, q):
    node = root
    while node:
        if p < node.val and q < node.val:
            node = node.left
        elif p > node.val and q > node.val:
            node = node.right
        else:
            return node.val
    return None

bst = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
assert lowest_common_ancestor(bst, 2, 8) == 6
assert lowest_common_ancestor(bst, 2, 4) == 2
print("LCA ")

LCA 


## Pattern 8 - Heap / Priority Queue
**Recognise it:** "k-th largest/smallest", "top k", "k closest", "merge k sorted", "median of stream".
Trick: keep a heap of **size k** → O(n log k).

### 30. Kth Largest Element in an Array - LC #215 (Medium) **Idea:** min-heap of the k largest seen so far; its top is the k-th largest.

In [31]:
import heapq

def find_kth_largest(nums, k):
    heap = []
    for x in nums:
        heapq.heappush(heap, x)
        if len(heap) > k:
            heapq.heappop(heap)      # drop the smallest
    return heap[0]

# Time O(n log k), Space O(k)
assert find_kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4
print("Kth Largest ")

Kth Largest 


### 31. K Closest Points to Origin - LC #973 (Medium)
**Idea:** max-heap of size k using negative distances (no need for the square root).

In [32]:
import heapq

def k_closest(points, k):
    heap = []                                   # (-dist, x, y)
    for x, y in points:
        heapq.heappush(heap, (-(x * x + y * y), x, y))
        if len(heap) > k:
            heapq.heappop(heap)                 # removes the farthest
    return [[x, y] for _, x, y in heap]

assert k_closest([[1, 3], [-2, 2]], 1) == [[-2, 2]]
assert sorted(k_closest([[3, 3], [5, -1], [-2, 4]], 2)) == [[-2, 4], [3, 3]]
print("K Closest ")

K Closest 


## Pattern 9 - Intervals
**Recognise it:** `[start, end]` pairs, meetings, overlaps. **Almost always: sort by start first.**

### 32. Merge Intervals - LC #56 (Medium) 

In [33]:
def merge_intervals(intervals):
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:                 # overlap
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

# Time O(n log n)
assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]
print("Merge Intervals ")

Merge Intervals 


## Pattern 10 - Graphs (DFS / BFS / Topological sort)
**Recognise it:** grids of cells, "islands", "connected", prerequisites / dependencies, shortest path in unweighted graph (BFS).
Build the graph with `defaultdict(list)`, track `visited` with a set.

### 33. Number of Islands - LC #200 (Medium) most asked graph question
**Idea:** scan the grid; at each unvisited `'1'`, count an island and flood-fill (DFS) all connected land so it isn't counted again.

In [34]:
def num_islands(grid):
    rows, cols = len(grid), len(grid[0])
    visited = set()

    def dfs(r, c):
        if (r < 0 or c < 0 or r >= rows or c >= cols
                or grid[r][c] == "0" or (r, c) in visited):
            return
        visited.add((r, c))
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:   # 4 directions
            dfs(r + dr, c + dc)

    islands = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in visited:
                islands += 1
                dfs(r, c)
    return islands

# Time O(rows * cols)
grid = [
    ["1", "1", "0", "0", "0"],
    ["1", "1", "0", "0", "0"],
    ["0", "0", "1", "0", "0"],
    ["0", "0", "0", "1", "1"],
]
assert num_islands(grid) == 3
print("Number of Islands ")

Number of Islands 


### 34. Rotting Oranges - LC #994 (Medium) - multi-source BFS
Each minute, rotten oranges (2) rot their fresh neighbours (1). Minutes until none fresh, or -1.

**Idea:** put **all** rotten oranges in the queue at once; BFS level = 1 minute.

In [35]:
from collections import deque

def oranges_rotting(grid):
    rows, cols = len(grid), len(grid[0])
    q = deque()
    fresh = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 2:
                q.append((r, c))
            elif grid[r][c] == 1:
                fresh += 1

    minutes = 0
    while q and fresh > 0:
        for _ in range(len(q)):
            r, c = q.popleft()
            for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    q.append((nr, nc))
        minutes += 1
    return minutes if fresh == 0 else -1

assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
assert oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1
assert oranges_rotting([[0, 2]]) == 0
print("Rotting Oranges ")

Rotting Oranges 


### 35. Course Schedule - LC #207 (Medium) - topological sort (Kahn's algorithm)
Can you finish all courses given prerequisite pairs `[course, prereq]`? = Does the graph have **no cycle**?

**Idea:** count incoming edges (in-degree). Start with courses that have 0 prerequisites; "take" them and decrease neighbours' in-degree. If we take all courses → no cycle.

In [36]:
from collections import defaultdict, deque

def can_finish(num_courses, prerequisites):
    graph = defaultdict(list)
    indegree = [0] * num_courses
    for course, prereq in prerequisites:
        graph[prereq].append(course)
        indegree[course] += 1

    q = deque(i for i in range(num_courses) if indegree[i] == 0)
    taken = 0
    while q:
        node = q.popleft()
        taken += 1
        for nxt in graph[node]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                q.append(nxt)
    return taken == num_courses

# Time O(V + E)
assert can_finish(2, [[1, 0]])
assert not can_finish(2, [[1, 0], [0, 1]])
print("Course Schedule ")

Course Schedule 


## Pattern 11 - Backtracking
**Recognise it:** "generate **all** subsets / permutations / combinations", "all ways to…". Template: **choose → explore (recurse) → un-choose**.

### 36. Subsets - LC #78 (Medium) 

In [37]:
def subsets(nums):
    result, path = [], []

    def backtrack(i):
        if i == len(nums):
            result.append(path[:])        # copy!
            return
        path.append(nums[i])              # choose nums[i]
        backtrack(i + 1)
        path.pop()                        # un-choose
        backtrack(i + 1)                  # skip nums[i]

    backtrack(0)
    return result

# Time O(n * 2^n)
assert sorted(subsets([1, 2, 3])) == sorted([[], [1], [2], [3], [1, 2], [1, 3], [2, 3], [1, 2, 3]])
print("Subsets ", subsets([1, 2, 3]))

Subsets  [[1, 2, 3], [1, 2], [1, 3], [1], [2, 3], [2], [3], []]


### 37. Permutations - LC #46 (Medium)

In [38]:
def permute(nums):
    result, path, used = [], [], [False] * len(nums)

    def backtrack():
        if len(path) == len(nums):
            result.append(path[:])
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True;  path.append(nums[i])     # choose
            backtrack()                                # explore
            used[i] = False; path.pop()                # un-choose

    backtrack()
    return result

# Time O(n * n!)
assert len(permute([1, 2, 3])) == 6
assert sorted(permute([0, 1])) == [[0, 1], [1, 0]]
print("Permutations ")

Permutations 


### 38. Combination Sum - LC #39 (Medium)
Unique combinations summing to `target`; each number may be reused.

In [39]:
def combination_sum(candidates, target):
    result, path = [], []

    def backtrack(start, remaining):
        if remaining == 0:
            result.append(path[:])
            return
        if remaining < 0:
            return
        for i in range(start, len(candidates)):   # start -> no duplicate orderings
            path.append(candidates[i])
            backtrack(i, remaining - candidates[i])  # i (not i+1): reuse allowed
            path.pop()

    backtrack(0, target)
    return result

assert sorted(combination_sum([2, 3, 6, 7], 7)) == [[2, 2, 3], [7]]
assert sorted(combination_sum([2, 3, 5], 8)) == [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
print("Combination Sum ")

Combination Sum 


## Pattern 12 - Dynamic Programming
**Recognise it:** "number of ways", "min / max cost", "can you reach", choices at each step, overlapping sub-problems.
Steps:
1. Define the state: `dp[i]` = answer for the first `i` items / for amount `i`…
2. Write the recurrence (how `dp[i]` comes from smaller states).
3. Base cases. 4. Order of computation (or just use `@cache` recursion).

### 39. Climbing Stairs - LC #70 (Easy) the "hello world" of DP
1 or 2 steps at a time; ways to reach step `n`? → `ways(n) = ways(n-1) + ways(n-2)` (Fibonacci).

In [40]:
def climb_stairs(n):
    one, two = 1, 1          # ways to reach step i-1 and i-2
    for _ in range(n - 1):
        one, two = one + two, one
    return one

assert [climb_stairs(n) for n in range(1, 6)] == [1, 2, 3, 5, 8]
print("Climbing Stairs ")

Climbing Stairs 


### 40. House Robber - LC #198 (Medium) Can't rob two adjacent houses. Max money?

**Recurrence:** `best(i) = max(best(i-1), best(i-2) + nums[i])` - skip this house, or rob it.

In [41]:
def rob(nums):
    prev2, prev1 = 0, 0          # best up to i-2, best up to i-1
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1

assert rob([1, 2, 3, 1]) == 4
assert rob([2, 7, 9, 3, 1]) == 12
print("House Robber ")

House Robber 


### 41. Coin Change - LC #322 (Medium) Fewest coins to make `amount` (or -1).

**State:** `dp[a]` = fewest coins for amount `a`. **Recurrence:** `dp[a] = min(dp[a - coin] + 1)` over all coins.

In [42]:
def coin_change(coins, amount):
    INF = float('inf')
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for coin in coins:
            if coin <= a:
                dp[a] = min(dp[a], dp[a - coin] + 1)
    return dp[amount] if dp[amount] != INF else -1

# Time O(amount * len(coins))
assert coin_change([1, 2, 5], 11) == 3       # 5 + 5 + 1
assert coin_change([2], 3) == -1
assert coin_change([1], 0) == 0
print("Coin Change ")

Coin Change 


### 42. Longest Increasing Subsequence - LC #300 (Medium)
**O(n²) DP:** `dp[i]` = LIS ending at `i` = 1 + max `dp[j]` for `j < i` with `nums[j] < nums[i]`.
**O(n log n):** keep `tails` (smallest tail for each length) and `bisect` into it.

In [43]:
import bisect

def length_of_lis(nums):
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp, default=0)

def length_of_lis_fast(nums):
    tails = []
    for x in nums:
        i = bisect.bisect_left(tails, x)
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)

for f in (length_of_lis, length_of_lis_fast):
    assert f([10, 9, 2, 5, 3, 7, 101, 18]) == 4
    assert f([0, 1, 0, 3, 2, 3]) == 4
    assert f([7, 7, 7, 7]) == 1
print("LIS ")

LIS 


### 43. Longest Common Subsequence - LC #1143 (Medium) - 2D DP **State:** `dp[i][j]` = LCS of `text1[:i]` and `text2[:j]`.
- chars equal → `dp[i-1][j-1] + 1`
- otherwise → `max(dp[i-1][j], dp[i][j-1])`

In [44]:
def longest_common_subsequence(text1, text2):
    m, n = len(text1), len(text2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i - 1] == text2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

# Time O(m * n)
assert longest_common_subsequence("abcde", "ace") == 3
assert longest_common_subsequence("abc", "def") == 0
print("LCS ")

LCS 


### 44. Maximum Subarray - LC #53 (Medium) - Kadane's algorithm
**Idea:** at each element: either extend the previous subarray or start fresh here - whichever is bigger.

In [45]:
def max_subarray(nums):
    current = best = nums[0]
    for x in nums[1:]:
        current = max(x, current + x)
        best = max(best, current)
    return best

assert max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
assert max_subarray([-1]) == -1
assert max_subarray([5, 4, -1, 7, 8]) == 23
print("Kadane ")

Kadane 


---
# Part 2 - Cheat Sheets

## 3.1 Which pattern? (read the question for these clues)

| Clue in the question | Try this |
|---|---|
| "pair / duplicate / count / seen before", O(n) | **Hash map / set** |
| sorted array, palindrome, pair sum | **Two pointers** |
| longest/shortest **substring / subarray** | **Sliding window** |
| brackets, "next greater element", undo | **Stack** (monotonic stack) |
| sorted + O(log n), "minimum value that works" | **Binary search** |
| linked list, middle, cycle | **Slow/fast pointers, dummy node** |
| tree | **DFS recursion** (or BFS for levels) |
| k-th largest, top k, closest k | **Heap of size k** |
| intervals / meetings | **Sort by start** |
| grid, islands, connections | **DFS / BFS on graph** |
| shortest path, unweighted | **BFS** |
| prerequisites / ordering | **Topological sort** |
| "all combinations / permutations / subsets" | **Backtracking** |
| "number of ways", "min/max cost", choices | **Dynamic programming** |

## 3.2 Big-O you must know

| Complexity | Name | Typical example | Max n (~1 sec) |
|---|---|---|---|
| O(1) | constant | dict lookup | any |
| O(log n) | logarithmic | binary search | any |
| O(n) | linear | one loop | ~10⁷ |
| O(n log n) | | sorting | ~10⁶ |
| O(n²) | quadratic | double loop | ~10⁴ |
| O(2ⁿ) | exponential | subsets | ~20 |
| O(n!) | factorial | permutations | ~10 |

> Look at the **constraints** on LeetCode: `n ≤ 10^5` means O(n²) is too slow → you need O(n) or O(n log n).

## 3.3 Interview routine (say this out loud)
1. **Clarify:** inputs, edge cases (empty, negatives, duplicates).
2. **Brute force** first + its complexity.
3. **Optimise** using a pattern from 3.1.
4. **Code** cleanly with good names.
5. **Test** with an example + an edge case.
6. **State complexity:** time and space.

---
# Part 3 - Practice (solve these yourself!)
Each stub has tests. Replace `pass` with your solution and run the cell. If you get stuck, the matching pattern is in the hint and the full problem is on LeetCode.

### Majority Element - LC #169 (Easy)
**Hint:** Hash map / Counter (bonus: Boyer-Moore O(1) space)

In [46]:
def majority_element(nums):
    pass

# --- tests: uncomment when ready ---
# assert majority_element([3, 2, 3]) == 3
# assert majority_element([2, 2, 1, 1, 1, 2, 2]) == 2

### Move Zeroes - LC #283 (Easy)
**Hint:** Two pointers (write index)

In [47]:
def move_zeroes(nums):
    # modify nums in place, return nothing
    pass

# --- tests: uncomment when ready ---
# a = [0, 1, 0, 3, 12]; move_zeroes(a); assert a == [1, 3, 12, 0, 0]

### Maximum Average Subarray I - LC #643 (Easy)
**Hint:** Fixed-size sliding window

In [48]:
def find_max_average(nums, k):
    pass

# --- tests: uncomment when ready ---
# assert find_max_average([1, 12, -5, -6, 50, 3], 4) == 12.75

### Remove Duplicates from Sorted Array - LC #26 (Easy)
**Hint:** Two pointers: a write index for the next unique value

In [49]:
def remove_duplicates(nums):
    # modify nums in place, return k = number of unique values
    pass

# --- tests: uncomment when ready ---
# a = [1, 1, 2]; k = remove_duplicates(a); assert k == 2 and a[:k] == [1, 2]
# b = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]; k = remove_duplicates(b); assert k == 5 and b[:k] == [0, 1, 2, 3, 4]

### Same Tree - LC #100 (Easy)
**Hint:** DFS recursion on two trees at once

In [50]:
def is_same_tree(p, q):
    pass

# --- tests: uncomment when ready ---
# assert is_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3]))
# assert not is_same_tree(build_tree([1, 2]), build_tree([1, None, 2]))

### Happy Number - LC #202 (Easy)
**Hint:** Hash set to detect a cycle (or slow/fast pointers)

In [51]:
def is_happy(n):
    pass

# --- tests: uncomment when ready ---
# assert is_happy(19) is True
# assert is_happy(2) is False

### House Robber II - LC #213 (Medium)
**Hint:** Houses in a circle: run House Robber twice (skip first house / skip last house)

In [52]:
def rob_circle(nums):
    pass

# --- tests: uncomment when ready ---
# assert rob_circle([2, 3, 2]) == 3
# assert rob_circle([1, 2, 3, 1]) == 4
# assert rob_circle([1, 2, 3]) == 3

### Find the Duplicate Number - LC #287 (Medium)
**Hint:** Floyd's cycle detection on indices (or binary search on the value range)

In [53]:
def find_duplicate(nums):
    pass

# --- tests: uncomment when ready ---
# assert find_duplicate([1, 3, 4, 2, 2]) == 2
# assert find_duplicate([3, 1, 3, 4, 2]) == 3

---
## What next?
0. Go back to any `01` / `02` notebook whose pattern felt weak.
1. Re-solve every problem in Part 2 on LeetCode **without looking** (numbers are in each title).
2. Continue with the **NeetCode 150** list (neetcode.io) - same pattern order as this notebook.
3. Aim for 1–2 problems per day; revisit any you couldn't solve after 3 days.

Good luck with your interviews! 